# ARISE — 05 · Visualisations & Dashboard Export

This notebook produces the artefacts the **React dashboard** consumes and shows
the key visualisations: PCA embedding map, Precision@K curves, and the
substructure breakdown. It writes `dashboard/public/data/<dataset>.json`.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))  # make the `arise` package importable
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline


In [ ]:
from arise.data import load_dataset, inject_anomalies, PAPER_INJECTION
from arise.pipeline import run_arise
from arise.export import build_dashboard_payload

graph = load_dataset("cora")
graph = inject_anomalies(graph, seed=42, **PAPER_INJECTION["Cora"])
result, model = run_arise(graph, alpha=0.8, epochs=60, lr=0.003,
                          weight_decay=1e-5, attr_rounds=4, seed=42,
                          return_model=True, verbose=False)
payload = build_dashboard_payload(graph, result, model=model)
print("payload keys:", list(payload.keys()))

## 1. PCA embedding map
Nodes coloured by true type. Anomalies should sit apart from the normal manifold.

In [ ]:
import numpy as np
emb2d = payload["embedding_2d"]
xs = np.array([p["x"] for p in emb2d]); ys = np.array([p["y"] for p in emb2d])
types = np.array([p["type"] for p in emb2d])
cmap = {"normal":"#9aa7bd", "topology":"#ff5c5c", "attribute":"#ffb454"}
plt.figure(figsize=(7,6))
for t in ["normal","topology","attribute"]:
    m = types==t
    plt.scatter(xs[m], ys[m], s=12, c=cmap[t], label=t, alpha=.7)
plt.legend(); plt.title("PCA of ARISE node embeddings"); plt.xlabel("PC1"); plt.ylabel("PC2"); plt.show()

## 2. Precision@K (paper Fig. 4)

In [ ]:
pk = payload["precision_at_k_curve"]
plt.figure(figsize=(7,4))
plt.plot(pk["ks"], pk["total"], "-o", label="total", color="#ff5c5c")
plt.plot(pk["ks"], pk["topology"], "-o", label="topology", color="#6ea8ff")
plt.plot(pk["ks"], pk["attribute"], "-o", label="attribute", color="#ffb454")
plt.xlabel("K"); plt.ylabel("Precision@K"); plt.legend(); plt.title("Precision@K"); plt.show()

## 3. Substructure breakdown (first detection round)

In [ ]:
r0 = payload["substructure_rounds"][0]
print(f"k={r0['k']}  #substructures={r0['num_substructures']}")
print(f"{'size':>5} {'avg_sim':>8} {'estimate':>9}")
for s in r0["substructures"][:10]:
    print(f"{s['size']:>5} {s['avg_similarity']:>8} {s['anomaly_estimate']:>9}")

## 4. Export JSON for the dashboard
Writes both `dashboard/public/data/` and `results/`, plus updates `index.json`.

In [ ]:
import json, os
os.makedirs("../dashboard/public/data", exist_ok=True)
os.makedirs("../results", exist_ok=True)
slug = payload["dataset"]["name"].lower()
for d in ["../dashboard/public/data", "../results"]:
    with open(f"{d}/{slug}.json", "w") as f:
        json.dump(payload, f)
print("wrote", slug, "AUC=", payload["metrics"]["overall"]["auc"])

# Tip: to regenerate ALL datasets at once, run from the project root:
#     python generate_results.py --datasets Cora Synthetic

## Summary
- The PCA map shows injected anomalies separating from the normal manifold.
- Precision@K and substructure tables match the paper's evaluation style.
- Results are exported to JSON; launch the **React dashboard** (`dashboard/`) to explore them interactively.

```bash
cd dashboard && npm install && npm run dev
```